In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

catalog_name = 'ecommerce'

In [0]:
df_bronze_brands = spark.table(f"{catalog_name}.bronze.brz_brands")

In [0]:
# 1.1 transformation: slv_brands -> brand_code

df_silver_brands = df_bronze_brands.withColumn("brand_code", F.upper(F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', '')))

df_silver_brands = df_silver_brands.dropDuplicates(["brand_code"])

In [0]:
# 1.2 Validation: slv_brands - brand_code

df_silver_brands.groupBy("brand_code") \
.count() \
.filter(F.col("count") > 1) \
.show()

In [0]:
# 2.1 transformation: slv_brands - brand_name

df_silver_brands = df_silver_brands.withColumn("brand_name", F.trim(F.col("brand_name")))

In [0]:
# 2.2 Validation: slv_brands - brand_name

df_silver_brands.groupBy("brand_name") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

In [0]:
# 3.1 transformation: slv_brands - category_code

df_silver_brands = df_silver_brands.withColumn("category_code", F.upper(F.col("category_code")))

category_anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

df_silver_brands = df_silver_brands.replace(category_anomalies, subset = "category_code")



In [0]:
# 3.2 Validation: brz_brands - category_code

valid_categories = [
    row["category_code"]
    for row in spark.table(f"{catalog_name}.silver.slv_categories_clean")
    .select("category_code") 
    .distinct()
    .collect()
]


df_silver_brands.filter(~F.col("category_code").isin(valid_categories)) \
    .select("category_code") \
    .show()


In [0]:
# 4.1 Quarantine Bad Data -> df_silver_brands_clean & df_silver_brands_quarantine

df_silver_brands_clean = df_silver_brands.filter(
    F.col("brand_code").isNotNull() & (F.col("brand_code") != "") &
    F.col("brand_name").isNotNull() & (F.col("brand_name") != "") &
    F.col("category_code").isNotNull() & (F.col("category_code") != "") &
    F.col("category_code").isin(valid_categories))

df_silver_brands_quarantine = df_silver_brands.filter(
    F.col("brand_code").isNull() | (F.col("brand_code") == "") |
    F.col("brand_name").isNull() | (F.col("brand_name") == "") |
    F.col("category_code").isNull() | (F.col("category_code") == "") |
    ~F.col("category_code").isin(valid_categories)) \
    .withColumn("rejection_reason", 
        F.when(F.col("brand_code").isNull() | (F.col("brand_code") == ""), "null or empty brand_code")
        .when(F.col("brand_name").isNull() | (F.col("brand_name") == ""), "null or empty brand_name")
        .when(F.col("category_code").isNull() | (F.col("category_code") == ""), "null or empty category_code")
        .otherwise("invalid category_code")
        )

In [0]:
# 4.2 Check Quarantined Brands

df_silver_brands_quarantine.show()

In [0]:
# 5.1 Write Up to Delta -> df_silver_brands_clean & df_silver_brands_quarantine

df_silver_brands_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands_clean")

df_silver_brands_quarantine.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands_quarantine")